# MES Workflow

This notebook explains MES execution with database-driven ERP integration and OPC-UA PLC control.

## Implemented Features and Specification Mapping

### Requirement 4.1 — MES Orders and Plant-Floor Management
- MES reads executable production lines from `mes_db.Production_Orders`.
- MES uses recipe/scheduling logic to decide operation sequence and dispatches machine actions.
- MES communicates with PLC through OPC-UA variable writes/reads via `plc_client.py`.

### Requirement 4.2 — MES User Interface
- `mes_monitor.py` provides order monitoring with:
  - current production list
  - completed order visibility
  - produced and pending quantities
  - algorithm status panel (priority ordering)
  - production time estimate per order

### Requirement 4.3 — Statistics
- Machine statistics are persisted and displayed:
  - total operating time
  - occupation indicator
  - tool changes
  - processed pieces (total/by type)
- Tool statistics:
  - operating time by tool
- Unloading statistics:
  - totals by dock
  - totals by dock and piece type

## Component Structure
- `MES/mes_server.py`: MES runtime entrypoint (database-to-control flow).
- `MES/mes_service.py`: core execution loop, status updates, and statistics persistence.
- `MES/mes_recipe_engine.py`: piece transformation and order-priority logic.
- `MES/plc_client.py`: OPC-UA node mapping and PLC command primitives.
- `MES/mes_monitor.py`: Streamlit monitoring interface.
- `MES/run_mes_programs.py`: convenience launcher.

## MES Execution Flow
1. MES polls `Production_Orders` for open production lines.
2. Recipe engine selects next action, respecting priority rules.
3. MES writes mapped OPC-UA command variables to PLC.
4. MES updates production progress and statistics tables.
5. MES monitor renders real-time state and KPI panels from database data.

## OPC-UA Variables Required by MES

The MES expects the following PLC symbols to be exposed in CODESYS Symbol Configuration.

### Create Piece (MES -> PLC)
- `Create_Piece_I1`, `Create_Piece_I2`, `Create_Piece_I3`, `Create_Piece_I4`, `Create_Piece_I5`
- `piece_material_I1`, `piece_material_I2`, `piece_material_I3`, `piece_material_I4`, `piece_material_I5`

### Tool Configuration (MES -> PLC)
Tool lists per machine (GVL arrays):
- `Tool_M1a[0..2]`, `Tool_M1b[0..2]`, `Tool_M1c[0..2]`
- `Tool_M2a[0..2]`, `Tool_M2b[0..2]`, `Tool_M2c[0..2]`
- `Tool_M3a[0..2]`, `Tool_M3b[0..2]`, `Tool_M3c[0..2]`
- `Tool_M4a[0..2]`, `Tool_M4b[0..2]`, `Tool_M4c[0..2]`

Tool timing table (GVL array):
- `Tool_Time[0..11]` (TIME values)

### Machine Commands (MES -> PLC)
Each Machine_Conveyer instance uses the following fields:
- `C6.Action_Type`, `C6.Process_Time`, `C6.Target_ID`, `C6.Needed_Tool`, `C6.Req_Top_ID`, `C6.Req_Leg_ID`, `C6.Req_Join_Tool`
- `C8.Action_Type`, `C8.Process_Time`, `C8.Target_ID`, `C8.Needed_Tool`, `C8.Req_Top_ID`, `C8.Req_Leg_ID`, `C8.Req_Join_Tool`
- `C10.Action_Type`, `C10.Process_Time`, `C10.Target_ID`, `C10.Needed_Tool`, `C10.Req_Top_ID`, `C10.Req_Leg_ID`, `C10.Req_Join_Tool`
- `C11.Action_Type`, `C11.Process_Time`, `C11.Target_ID`, `C11.Needed_Tool`, `C11.Req_Top_ID`, `C11.Req_Leg_ID`, `C11.Req_Join_Tool`
- `C13.Action_Type`, `C13.Process_Time`, `C13.Target_ID`, `C13.Needed_Tool`, `C13.Req_Top_ID`, `C13.Req_Leg_ID`, `C13.Req_Join_Tool`
- `C15.Action_Type`, `C15.Process_Time`, `C15.Target_ID`, `C15.Needed_Tool`, `C15.Req_Top_ID`, `C15.Req_Leg_ID`, `C15.Req_Join_Tool`
- `C16.Action_Type`, `C16.Process_Time`, `C16.Target_ID`, `C16.Needed_Tool`, `C16.Req_Top_ID`, `C16.Req_Leg_ID`, `C16.Req_Join_Tool`
- `C18.Action_Type`, `C18.Process_Time`, `C18.Target_ID`, `C18.Needed_Tool`, `C18.Req_Top_ID`, `C18.Req_Leg_ID`, `C18.Req_Join_Tool`
- `C20.Action_Type`, `C20.Process_Time`, `C20.Target_ID`, `C20.Needed_Tool`, `C20.Req_Top_ID`, `C20.Req_Leg_ID`, `C20.Req_Join_Tool`
- `C21.Action_Type`, `C21.Process_Time`, `C21.Target_ID`, `C21.Needed_Tool`, `C21.Req_Top_ID`, `C21.Req_Leg_ID`, `C21.Req_Join_Tool`
- `C23.Action_Type`, `C23.Process_Time`, `C23.Target_ID`, `C23.Needed_Tool`, `C23.Req_Top_ID`, `C23.Req_Leg_ID`, `C23.Req_Join_Tool`
- `C25.Action_Type`, `C25.Process_Time`, `C25.Target_ID`, `C25.Needed_Tool`, `C25.Req_Top_ID`, `C25.Req_Leg_ID`, `C25.Req_Join_Tool`

### Warehouse Counters (MES <- PLC)
- `W1_Count`, `W2_Count`

### Unload Requests (MES -> PLC)
- `Unload_Req_U1` .. `Unload_Req_U10`
- `piece_material_U1` .. `piece_material_U10`

### Unload Snapshot (MES <- PLC)
- `Unload_Req_U1` .. `Unload_Req_U10`
- `piece_material_U1` .. `piece_material_U10`

These symbols must be exposed under the GVL or PLC_PRG nodes in OPC-UA.

### MES launcher
This code block starts the full MES program stack and opens the graphical MES monitor in your browser. The terminal only launches the service; the interface itself is a web GUI.

In [ ]:
from pathlib import Path
import subprocess, sys

root = Path.cwd()
mes_dir = root / 'MES'

def run_mes_mode(mode='all'):
    cmd = [sys.executable, str(mes_dir / 'run_mes_programs.py'), '--mode', mode]
    print('Running:', ' '.join(cmd))
    print('Modes: server, service, monitor, all')
    print('The MES monitor will open in your browser on port 8502.')
    return subprocess.Popen(cmd)

# Run the full MES component from this notebook:
# p_all = run_mes_mode('all')
# p_monitor = run_mes_mode('monitor')